# 06 - Field Data Example: Well 1 Analog

This notebook applies the toolkit to a realistic field-analog DFIT built from
published parameters from Siddiqui & Qureshi (DFITAnalysis, MPL v2.0):
pump time 518 seconds, tight gas formation, normal leakoff.

Unlike the clean synthetic tests in earlier notebooks, this dataset has
realistic gauge noise, a pump time that is not a round number, and the
full range of wellbore-storage effects in the early falloff. It is
processed exactly as a field analyst would work it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dfit

## 1. Load the data

The CSV has four columns: time_min, pressure_psi, rate_bpm, G.
The pump time is 518 s = 8.63 min, inferred automatically from the rate column.

In [ ]:
df = pd.read_csv("data/well1_field_analog.csv")
print(df.shape)
print(df.head())
print()
print("Pump time inferred from rate column: last nonzero rate at",
      df.loc[df.rate_bpm>0, "time_min"].max(), "min")

## 2. Injection + early falloff overview

In [ ]:
fig, (ax1,ax2) = plt.subplots(2,1,figsize=(9,6),sharex=False)
# injection + falloff pressure
ax1.plot(df.time_min, df.pressure_psi, lw=1, color="steelblue")
ax1.axvline(df.loc[df.rate_bpm>0,"time_min"].max(), color="k",
            ls="--", lw=1, label="shut-in")
ax1.set_ylabel("Pressure (psi)"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title("Well 1 Analog - DFIT pressure record")
# rate
ax2.fill_between(df.time_min, df.rate_bpm, alpha=0.4, color="orange")
ax2.set_xlabel("Time (min)"); ax2.set_ylabel("Rate (bbl/min)")
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. ISIP estimation

With 3.5 psi gauge noise and real wellbore storage, the ISIP is not simply
the pressure at the moment the rate hits zero. The log-time extrapolation
method corrects for the early decompression period.

In [ ]:
from dfit import isip_log_extrapolation
t_pump = df.loc[df.rate_bpm>0,"time_min"].max()
isip_res = isip_log_extrapolation(df.time_min.values, df.pressure_psi.values, t_pump)
print(f"ISIP estimate : {isip_res.isip:,.0f} psi")
print(f"Method        : {isip_res.method}")
print(f"Note          : {isip_res.note}")

## 4. G-function diagnostic plot

In [ ]:
from dfit.derivatives import semilog_derivative, first_derivative
from dfit.gfunction import G_from_time

G = G_from_time(df.time_min.values, t_pump)
fall = np.isfinite(G)
Gf = G[fall]; pf = df.pressure_psi.values[fall]
o = np.argsort(Gf); Gf,pf = Gf[o],pf[o]

sgd = semilog_derivative(Gf, pf, smooth=7)
early = (Gf>0)&(Gf<=0.35*Gf.max())
slope = np.sum(Gf[early]*sgd[early])/np.sum(Gf[early]**2)

fig,ax1 = plt.subplots(figsize=(9,5))
ax1.plot(Gf, pf, "b-", lw=1, label="Pressure")
ax1.set_xlabel("G-function"); ax1.set_ylabel("Pressure (psi)",color="b")
ax2 = ax1.twinx()
ax2.plot(Gf, sgd, "r-", lw=1.3, label="G*dP/dG")
ax2.plot(Gf, slope*Gf, "k--", lw=1, label="Reference line")
ax2.set_ylabel("G*dP/dG (psi)",color="r")
ax1.set_title("Well 1 Analog - G-function diagnostic")
lines=ax1.get_lines()+ax2.get_lines()
ax1.legend(lines,[l.get_label() for l in lines],fontsize=8)
plt.tight_layout(); plt.show()

## 5. Full automated interpretation

In [ ]:
res = dfit.analyze_dfit(df.time_min.values, df.pressure_psi.values, df.rate_bpm.values)
print(res.summary())
print()
for k,v in res.notes.items():
    print(f"[{k}] {v}")

## 6. Fracture design parameters

In [ ]:
from dfit.fracdesign import fracture_design_from_dfit, design_summary

design = fracture_design_from_dfit(
    isip_psi=res.isip_psi,
    closure_pressure_psi=res.closure_pressure_psi,
    closure_time_min=res.closure_time_min or 150.0,
    t_pump_min=t_pump,
    fracture_height_ft=80.0,
    youngs_modulus_psi=3.8e6,
    design_half_length_ft=400.0,
    design_rate_bpm=8.0,
    tvd_ft=7200.0,
)
print(design_summary(design))

### What we learned from this field analog

- Pump time 518 s (8.63 min) is short enough that the G-function analysis
  is clean but the after-closure reservoir transient does not fully develop.
- The ISIP log-extrapolation correctly removes the early decompression overshoot.
- The fracture design step directly consumes the DFIT outputs, closing the
  loop from test to treatment - the core value proposition of a DFIT.